# Usage of GC.Analysis.API for GC Analysis

In [1]:
// In order to use a custom TraceEvent library, we need to reference it directly here

//#r "C:\Dev\perfview\src\TraceEvent\bin\Release\netstandard2.0\Microsoft.Diagnostics.Tracing.TraceEvent.dll"
#r "C:\Users\musharm\source\repos\perfview\src\TraceEvent\bin\Release\netstandard2.0\Microsoft.Diagnostics.Tracing.TraceEvent.dll"

In [2]:
#r "nuget: XPlot.Plotly"
#r "nuget: XPlot.Plotly.Interactive"
#r "nuget: Microsoft.Data.Analysis"
#r "nuget: Newtonsoft.Json"

using Etlx = Microsoft.Diagnostics.Tracing.Etlx;
using Microsoft.Data.Analysis;
using Microsoft.Diagnostics.Tracing.Analysis.GC;
using Microsoft.Diagnostics.Tracing.Analysis;
using Microsoft.Diagnostics.Tracing.Parsers.Clr;
using Microsoft.Diagnostics.Tracing;
using XPlot.Plotly;

using System.IO;
using Newtonsoft.Json;

Installed Packages Microsoft.Data.Analysis, 0.21.1 Newtonsoft.Json, 13.0.3 XPlot.Plotly, 4.0.6 XPlot.Plotly.Interactive, 4.0.7

Loading extensions from `Q:\.tools\.nuget\packages\microsoft.data.analysis\0.21.1\interactive-extensions\dotnet\Microsoft.Data.Analysis.Interactive.dll`

Loading extensions from `Q:\.tools\.nuget\packages\xplot.plotly.interactive\4.0.7\lib\net7.0\XPlot.Plotly.Interactive.dll`

Configuring PowerShell Kernel for XPlot.Plotly integration.

Installed support for XPlot.Plotly.

In [3]:
System.Diagnostics.Process.GetCurrentProcess().Id

7132

## Building and Using The GC Analysis API

Before building `GC.Analysis.API`, make sure we change the project file so that it references the local copy instead of pulling it from Nuget. 

In [4]:
dotnet build -c Debug "..\..\GC.Analysis.API" /p:CustomTraceEvent=true /p:PerfViewPath=C:\Users\musharm\source\repos\perfview

  Determining projects to restore...
  All projects are up-to-date for restore.
  GC.Analysis.API -> C:\Users\musharm\source\repos\performance_dynamic\artifacts\bin\GC.Analysis.API\Debug\net8.0\GC.Analysis.API.dll

Build succeeded.
    0 Warning(s)
    0 Error(s)

Time Elapsed 00:00:01.26


In [5]:
#r "C:\Users\musharm\source\repos\performance_dynamic\artifacts\bin\GC.Analysis.API\Debug\net8.0\GC.Analysis.API.dll" 

using GC.Analysis.API;
using GC.Analysis.API.DynamicEvents;

## Creating the Analyzer

In [6]:
var TRACE_PATH  = @"C:\Users\musharm\source\repos\performance\src\benchmarks\gc\GC.Infrastructure\Configurations\ASPNetBenchmarks\Test_Logs\datas_0\ConnectionCloseHttpSys_Windows.gc.etl.zip";

### Get Analyzer From a Single Path

#### Get All Processes From a Trace

In [7]:
Analyzer gcTraceData = new Analyzer(tracePath: TRACE_PATH);

#### Get Select Processes From a Trace

In [8]:
gcTraceData.SummarizeTrace(processName: "Benchmarks")

index value 0 index Process ID: 5096 Start (ms) Start GC Index End (ms) End GC Index Notes 0 GC 11280.63 1 41456.72 950 950 GCs found for Process: 5096 1 CPU Samples -1 -1 No CPU Events Found for Process: 5096 2 CSwitch Data -1 -1 No CSwitch Events Found for Process: 5096.

#### Getting To a ``List<TraceGC>`` from the Trace Data For a Process

In [9]:
List<GCProcessData> corerunData    = gcTraceData.GetProcessGCData(processName: "Benchmarks");
GCProcessData corerunToInvestigate = corerunData[0];
List<TraceGC> traceGCs             = corerunToInvestigate.GCs;

# Dynamic Event

In [10]:
DynamicEventSchema.Set(new List<DynamicEventSchema>
{
    new DynamicEventSchema
    {
        DynamicEventName = "SizeAdaptationSample",
        Fields = new List<KeyValuePair<string, Type>>
        {
            new KeyValuePair<string, Type>("version", typeof(ushort)),
            new KeyValuePair<string, Type>("GCIndex", typeof(ulong)),
            new KeyValuePair<string, Type>("ElapsedTimeBetweenGCs", typeof(uint)),
            new KeyValuePair<string, Type>("GCPauseTime", typeof(uint)),
            new KeyValuePair<string, Type>("SOHMSLWaitTime", typeof(uint)),
            new KeyValuePair<string, Type>("UOHMSLWaitTime", typeof(uint)),
            new KeyValuePair<string, Type>("TotalSOHStableSize", typeof(ulong)),
            new KeyValuePair<string, Type>("Gen0BudgetPerHeap", typeof(uint)),
        },
        MinOccurrence = 0
    },
    new DynamicEventSchema
    {
        DynamicEventName = "SizeAdaptationTuning",
        Fields = new List<KeyValuePair<string, Type>>
        {
            new KeyValuePair<string, Type>("version", typeof(ushort)),
            new KeyValuePair<string, Type>("NewNHeaps", typeof(ushort)),
            new KeyValuePair<string, Type>("MaxHeapCountDatas", typeof(ushort)),
            new KeyValuePair<string, Type>("MinHeapCountDatas", typeof(ushort)),
            new KeyValuePair<string, Type>("CurrentGCIndex", typeof(ulong)),
            new KeyValuePair<string, Type>("TotalSOHStableSize", typeof(ulong)),
            new KeyValuePair<string, Type>("MedianThroughputCostPercent", typeof(float)),
            new KeyValuePair<string, Type>("TcpToConsider", typeof(float)),
            new KeyValuePair<string, Type>("CurrentAroundTargetAccumulation", typeof(float)),
            new KeyValuePair<string, Type>("RecordedTcpCount", typeof(ushort)),
            new KeyValuePair<string, Type>("RecordedTcpSlope", typeof(float)),
            new KeyValuePair<string, Type>("NumGcsSinceLastChange", typeof(uint)),
            new KeyValuePair<string, Type>("AggFactor", typeof(byte)),
            new KeyValuePair<string, Type>("ChangeDecision", typeof(ushort)),
            new KeyValuePair<string, Type>("AdjReason", typeof(ushort)),
            new KeyValuePair<string, Type>("HcChangeFreqFactor", typeof(ushort)),
            new KeyValuePair<string, Type>("HcFreqReason", typeof(ushort)),
            new KeyValuePair<string, Type>("AdjMetric", typeof(byte))
        },
        MinOccurrence = 0
    },
});

## Checks 

In [22]:
// From: "C:\Users\musharm\source\repos\performance\src\benchmarks\gc\GC.Infrastructure\Configurations\ASPNetBenchmarks\Test_Logs\datas_0\ConnectionCloseHttpSys_Windows_GCLog\gclog.log.log"

/*
[ 7664]SizeAdaptationSample: gc_index=2, elapsed_between_gcs=11039, gc_pause_time=3036, soh_msl_wait_time=0, uoh_msl_wait_time=0, total_soh_stable_size=50916, gen0_budget_per_heap=2621440
[ 7664]SizeAdaptationSample: gc_index=3, elapsed_between_gcs=9920, gc_pause_time=2289, soh_msl_wait_time=0, uoh_msl_wait_time=0, total_soh_stable_size=292624, gen0_budget_per_heap=2926240
[ 7664]SizeAdaptationSample: gc_index=4, elapsed_between_gcs=10586, gc_pause_time=2283, soh_msl_wait_time=0, uoh_msl_wait_time=0, total_soh_stable_size=531600, gen0_budget_per_heap=5316000
[ 7664]SizeAdaptationTuning: new_n_heaps: 2, max_heap_count_datas: 2, min_heap_count_datas: 0, current_gc_index: 4, total_soh_stable_size: 531600, median_throughput_cost_percent: 23.07, tcp_to_consider: 23.075, current_around_target_accumulation: 21.075, recorded_tcp_count: 1, recorded_tcp_slope: 0.000, num_gcs_since_last_change: 4, agg_factor: 1, change_decision: 0, adj_reason: 1, hc_change_freq_factor: 0, hc_freq_reason: 0, adj_metric: 2
*/

### Size Adaptation Sample

In [19]:
var d = (traceGCs.FirstOrDefault(t => t.Number == 2).DynamicEvents().SizeAdaptationSample);
Console.WriteLine($"GCIndex | Val: {d.GCIndex} | Expected: 2"); // ulong
Console.WriteLine($"GCPauseTime | Val: {d.GCPauseTime} | Expected: 3036"); // uint


GCIndex | Val: 2 | Expected: 2
GCPauseTime | Val: 3036 | Expected: 3036


### Size Adaptation Tuning

In [20]:
var d = (traceGCs.FirstOrDefault(t => t.Number == 4).DynamicEvents().SizeAdaptationTuning);
Console.WriteLine($"CurrentAroundTargetAccumulation | Val: {Math.Round((double)d.CurrentAroundTargetAccumulation, 3)} | Expected: {21.075}"); // float
Console.WriteLine($"AggFactor | Val: {d.AggFactor} | Expected: 1"); // byte
Console.WriteLine($"TcpToConsider | Val: {Math.Round((double)d.TcpToConsider, 3)} | Expected: {23.075}"); // float
Console.WriteLine($"MinHeapCountDatas | Val: {d.MinHeapCountDatas} | Expected: 0"); // ushort
Console.WriteLine($"TotalSOHStableSize | Val: {d.TotalSOHStableSize} | Expected: 531600"); // ushort

CurrentAroundTargetAccumulation | Val: 21.075 | Expected: 21.075
AggFactor | Val: 1 | Expected: 1
TcpToConsider | Val: 23.075 | Expected: 23.075
MinHeapCountDatas | Val: 0 | Expected: 0
TotalSOHStableSize | Val: 531600 | Expected: 531600
